In [1]:
!pip install -q torch transformers datasets tokenizers \
    sentence-transformers rank-bm25 pytrec-eval-terrier tqdm scipy comet-ml logging

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.2/786.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.7 MB/s eta 0:00:00


In [2]:
import os, sys, logging, json

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

In [3]:
!gdown --folder https://drive.google.com/drive/folders/1-NSm7k58UojqJEUmZztfruc_RXk3EacG?usp=sharing -O /kaggle/working/

Retrieving folder contents
Processing file 1yDE59CL6krlswPTPNsscAOLrUL7rxg3s doc_ids.json
Processing file 1MGSTENNRSa5J-qHAEbk6Evjex9RU7RV8 doc_lengths.npy
Processing file 1aETyJoY6BGbFvERSSDzZCD3kEUi8CVXc doc_offsets.npy
Processing file 1Iv_MsQfPV8vsR4b2g3U0UnDTj_zDwFf7 embeddings.npy
Processing file 1lNBzAVSxWdyYpgFVBbTJsgAwB98EMFoz metadata.json
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1yDE59CL6krlswPTPNsscAOLrUL7rxg3s
To: /kaggle/working/index_finetuned/doc_ids.json
100%|█████████████████████████████████████████| 444k/444k [00:00<00:00, 120MB/s]
Downloading...
From: https://drive.google.com/uc?id=1MGSTENNRSa5J-qHAEbk6Evjex9RU7RV8
To: /kaggle/working/index_finetuned/doc_lengths.npy
100%|█████████████████████████████████████████| 280k/280k [00:00<00:00, 101MB/s]
Downloading...
From: https://drive.google.com/uc?id=1aETyJoY6BGbFvERSSDzZCD3kEUi8CVXc
To: /kaggle/working/ind

In [4]:
!pip install -q mteb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 51.1 MB/s eta 0:00:00


In [5]:
from diploma_code import IRMetrics, SystemMetrics, evaluate_robustness
from diploma_code import EvalConfig

2026-04-18 21:23:58,968 NumExpr defaulting to 4 threads.
2026-04-18 21:24:00,307 TensorFlow version 2.19.0 available.
2026-04-18 21:24:00,312 JAX version 0.7.2 available.
2026-04-18 21:24:08.284796: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776547448.501228      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776547448.565716      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776547449.077959      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776547449.078003      22 computation_placer.cc:177] computation placer already

In [ ]:
qrels = {}
with open("/kaggle/input/datasets/vikakuznetzowa/diploma-ru-hard-negatives/qrels.tsv") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 4:
            qid, _, did, rel = parts[:4]
            qrels.setdefault(qid, {})[did] = int(rel)

In [7]:
from diploma_code import build_model_and_tokenizer
from diploma_code import ColBERTIndexer, ColBERTRetriever
from diploma_code import ModelConfig, IndexConfig
import torch

device = "cuda"

model_cfg = ModelConfig(encoder_name="xlm-roberta-base", embedding_dim=128)
model, tokenizer = build_model_and_tokenizer(model_cfg)

ckpt = torch.load("/kaggle/input/models/vikakuznetzowa/diploma-phase2/pytorch/default/1/checkpoints/checkpoint_phase2_best.pt", map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])
print("Fine-tuned model loaded.")

2026-04-18 21:24:23,160 HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
2026-04-18 21:24:23,233 HTTP Request: GET https://huggingface.co/xlm-roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

2026-04-18 21:24:23,316 HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-18 21:24:23,317 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-18 21:24:23,386 HTTP Request: GET https://huggingface.co/xlm-roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

2026-04-18 21:24:23,469 HTTP Request: GET https://huggingface.co/api/models/xlm-roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-18 21:24:23,535 HTTP Request: GET https://huggingface.co/api/models/FacebookAI/xlm-roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-18 21:24:23,596 HTTP Request: GET https://huggingface.co/api/models/xlm-roberta-base/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-18 21:24:23,665 HTTP Request: GET https://huggingface.co/api/models/FacebookAI/xlm-roberta-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-04-18 21:24:23,729 HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/sentencepiece.bpe.model "HTTP/1.1 200 OK"
2026-04-18 21:24:23,794 HTTP Request: GET https://huggingface.co/xlm-roberta-base/resolve/main/sentencepiece.bpe.model "HTTP/1.1 200 OK"


...

2026-04-18 21:24:24,505 HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
2026-04-18 21:24:24,593 HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/special_tokens_map.json "HTTP/1.1 404 Not Found"
2026-04-18 21:24:24,658 HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-04-18 21:24:27,283 HTTP Request: GET https://huggingface.co/api/models/xlm-roberta-base "HTTP/1.1 307 Temporary Redirect"
2026-04-18 21:24:27,346 HTTP Request: GET https://huggingface.co/api/models/FacebookAI/xlm-roberta-base "HTTP/1.1 200 OK"
2026-04-18 21:24:27,429 HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
2026-04-18 21:24:27,572 HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-04-18 21:24:27,664 HTTP Request: HEAD https://huggingf

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fine-tuned model loaded.


In [ ]:
retriever = ColBERTRetriever(
    model, tokenizer,
    index_dir="/kaggle/working/index_finetuned",
    model_cfg=model_cfg,
    device="cuda",
)

with open("/kaggle/input/datasets/vikakuznetzowa/diploma-ru-hard-negatives/queries.jsonl") as f:
    queries_data = [json.loads(line) for line in f]
queries_dict = {q["id"]: q["text"] for q in queries_data}

eval_cfg = EvalConfig(
    robustness_typo_rate=0.2,
    robustness_num_augments=4,
    k_values=[1, 5, 10, 20, 100],
)

2026-04-18 21:24:53,639 Loaded index: 35008 docs, 4754342 vectors


In [9]:
robustness = evaluate_robustness(
    retrieve_fn=lambda q: retriever.retrieve(q, top_k=100),
    queries=queries_dict,
    qrels=qrels,
    eval_cfg=eval_cfg,
)

In [ ]:
print("\n--- Robustness degradation (%) ---")
for metric, pct in robustness["degradation_pct"].items():
    print(f"  {metric}: -{pct:.1f}%")


--- Robustness degradation (%) ---
  ndcg_cut_10: -35.5%
  recall_5: -35.1%
  recip_rank: -35.8%
  ndcg_cut_1: -37.6%
  ndcg_cut_5: -36.1%
  recall_10: -33.7%
  ndcg_cut_20: -34.7%
  map_cut_20: -36.0%
  map_cut_100: -35.8%
  recall_1: -37.6%
  recall_100: -26.3%
  recall_20: -31.5%
  ndcg_cut_100: -33.3%
  map_cut_1: -37.6%
  map_cut_10: -36.3%
  map_cut_5: -36.6%
